# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [1]:
# Load the libraries as required.

%load_ext dotenv
%dotenv 
import os
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import PowerTransformer
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV
import pickle

In [2]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [3]:
X = fires_dt.drop('area', axis=1)
y = fires_dt['area']

In [10]:
print(X.head())
y.head()

   coord_x  coord_y month  day  ffmc   dmc     dc  isi  temp  rh  wind  rain
0        7        5   mar  fri  86.2  26.2   94.3  5.1   8.2  51   6.7   0.0
1        7        4   oct  tue  90.6  35.4  669.1  6.7  18.0  33   0.9   0.0
2        7        4   oct  sat  90.6  43.7  686.9  6.7  14.6  33   1.3   0.0
3        8        6   mar  fri  91.7  33.3   77.5  9.0   8.3  97   4.0   0.2
4        8        6   mar  sun  89.3  51.3  102.2  9.6  11.4  99   1.8   0.0


0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: area, dtype: float64

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [13]:

# Numeric and Categorical Features
numeric_features = ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']
categorical_features = ['month', 'day']

# preproc1: performing scaling and one-hot encoding
preproc1 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='drop'  
)

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [16]:
# preproc2: using PowerTransformer (which finds best transformation)
preproc2 = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('transform', PowerTransformer(method='yeo-johnson')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='drop'
)

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [24]:
# Pipeline A = preproc1 + baseline

pipeline_A = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', Ridge())
])

In [19]:
# Pipeline B = preproc2 + baseline

pipeline_B = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', Ridge())
])

In [20]:
# Pipeline C = preproc1 + advanced model (GradientBoostingRegressor)

pipeline_C = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', GradientBoostingRegressor(random_state=42))
])


In [25]:
# Pipeline D = preproc2 + advanced model (RandomForestRegressor)

from sklearn.ensemble import RandomForestRegressor

pipeline_D = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', RandomForestRegressor(random_state=42))
])

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [27]:
y_log = np.log1p(y)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

# Hyperparameter tuning for Pipeline A
param_grid_A = {
    'regressor__alpha': [0.1, 1.0, 10.0, 100.0],
    'regressor__solver': ['auto', 'svd', 'cholesky', 'lsqr']
}

grid_search_A = GridSearchCV(
    pipeline_A,
    param_grid_A,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

# Fit and evaluate Pipeline A
grid_search_A.fit(X_train, y_train)
print("Pipeline A (preproc1 + Ridge) :")
print(f"Best parameters: {grid_search_A.best_params_}")
print(f"Best CV score: {-grid_search_A.best_score_:.4f}")

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Pipeline A (preproc1 + Ridge) :
Best parameters: {'regressor__alpha': 100.0, 'regressor__solver': 'lsqr'}
Best CV score: 2.1236


In [30]:
# GridSearchCV for Pipeline B
# Use same parameter grid as Pipeline A
grid_search_B = GridSearchCV(
    pipeline_B,
    param_grid_A,  
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

# Fit and evaluate Pipeline B
grid_search_B.fit(X_train, y_train)
print("\nPipeline B (preproc2 + Ridge) :")
print(f"Best parameters: {grid_search_B.best_params_}")
print(f"Best CV score: {-grid_search_B.best_score_:.4f}")

Fitting 5 folds for each of 16 candidates, totalling 80 fits

Pipeline B (preproc2 + Ridge) :
Best parameters: {'regressor__alpha': 100.0, 'regressor__solver': 'lsqr'}
Best CV score: 1.9186


In [33]:
# Hyperparameter tuning for Pipeline C
param_grid_C = {
    'regressor__n_estimators': [100, 200],
    'regressor__learning_rate': [0.05, 0.1],
    'regressor__max_depth': [3, 5, 7],
    'regressor__min_samples_leaf': [1, 2, 4]
}

# GridSearchCV for Pipeline C
grid_search_C = GridSearchCV(
    estimator=pipeline_C,
    param_grid=param_grid_C,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

# Fit and evaluate Pipeline C
grid_search_C.fit(X_train, y_train)
print("\nPipeline C (preproc1 + advanced model (GradientBoostingRegressor)) :")
print(f"Best parameters: {grid_search_C.best_params_}")
print(f"Best CV score: {-grid_search_C.best_score_:.4f}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits

Pipeline C (preproc1 + advanced model (GradientBoostingRegressor)) :
Best parameters: {'regressor__learning_rate': 0.05, 'regressor__max_depth': 3, 'regressor__min_samples_leaf': 1, 'regressor__n_estimators': 100}
Best CV score: 2.0854


In [32]:
# Hyperparameter tuning for Pipeline D
param_grid_D = {
    'regressor__n_estimators': [50, 100, 200],  
    'regressor__max_depth': [5, 10, 20],  
    'regressor__min_samples_split': [2, 5, 10],  
    'regressor__min_samples_leaf': [1, 2] 
}

# GridSearchCV for Pipeline D
grid_search_D = GridSearchCV(
    estimator=pipeline_D,
    param_grid=param_grid_D,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

# Fit and evaluate Pipeline B
grid_search_D.fit(X_train, y_train)
print("\nPipeline D (preproc2 + advanced model (RandomForestRegressor)) :")
print(f"Best parameters: {grid_search_D.best_params_}")
print(f"Best CV score: {-grid_search_D.best_score_:.4f}")

Fitting 5 folds for each of 54 candidates, totalling 270 fits

Pipeline D (preproc2 + advanced model (RandomForestRegressor)) :
Best parameters: {'regressor__max_depth': 5, 'regressor__min_samples_leaf': 2, 'regressor__min_samples_split': 10, 'regressor__n_estimators': 200}
Best CV score: 2.0202


In [35]:
results_A = pd.DataFrame(grid_search_A.cv_results_)
print(results_A.head())

   mean_fit_time  std_fit_time  mean_score_time  std_score_time   
0       0.005151      0.000787         0.002326        0.000250  \
1       0.007822      0.002946         0.004775        0.001624   
2       0.004702      0.001084         0.002212        0.000707   
3       0.004863      0.000281         0.002010        0.000272   
4       0.006572      0.002255         0.002867        0.001698   

  param_regressor__alpha param_regressor__solver   
0                    0.1                    auto  \
1                    0.1                     svd   
2                    0.1                cholesky   
3                    0.1                    lsqr   
4                    1.0                    auto   

                                              params  split0_test_score   
0  {'regressor__alpha': 0.1, 'regressor__solver':...          -1.970132  \
1  {'regressor__alpha': 0.1, 'regressor__solver':...          -1.970132   
2  {'regressor__alpha': 0.1, 'regressor__solver':...       

# Evaluate

+ Which model has the best performance?

In [36]:
comparison_summary = pd.DataFrame({
    'Pipeline': ['A (preproc1 + Ridge)', 'B (preproc2 + Ridge)', 'C (preproc1 + GBR)', 'D (preproc2 + RF)'],
    'Best_CV_RMSE': [
        np.sqrt(-grid_search_A.best_score_),
        np.sqrt(-grid_search_B.best_score_),
        np.sqrt(-grid_search_C.best_score_),
        np.sqrt(-grid_search_D.best_score_)
    ],
    'Best_Params': [
        grid_search_A.best_params_,
        grid_search_B.best_params_,
        grid_search_C.best_params_,
        grid_search_D.best_params_,
    ]
})

print(comparison_summary)

               Pipeline  Best_CV_RMSE   
0  A (preproc1 + Ridge)      1.457256  \
1  B (preproc2 + Ridge)      1.385141   
2    C (preproc1 + GBR)      1.444105   
3     D (preproc2 + RF)      1.421350   

                                         Best_Params  
0  {'regressor__alpha': 100.0, 'regressor__solver...  
1  {'regressor__alpha': 100.0, 'regressor__solver...  
2  {'regressor__learning_rate': 0.05, 'regressor_...  
3  {'regressor__max_depth': 5, 'regressor__min_sa...  


In [39]:
# Determining the model performed best on the test set
test_scores = {
    'Pipeline_A': mean_squared_error(y_test, grid_search_A.best_estimator_.predict(X_test)),
    'Pipeline_B': mean_squared_error(y_test, grid_search_B.best_estimator_.predict(X_test)),
    'Pipeline_C': mean_squared_error(y_test, grid_search_C.best_estimator_.predict(X_test)),
    'Pipeline_D': mean_squared_error(y_test, grid_search_D.best_estimator_.predict(X_test))
}

# Determining the best model with lowest MSE
best_pipeline_name = min(test_scores, key=test_scores.get)
best_mse = test_scores[best_pipeline_name]
best_rmse = np.sqrt(best_mse)

print(f"Best performing model: {best_pipeline_name}")
print(f"Best RMSE: {best_rmse:.4f}")

Best performing model: Pipeline_B
Best RMSE: 1.4685


# Export

+ Save the best performing model to a pickle file.

In [38]:
# Determining the model performed best on the test set
test_scores = {
    'Pipeline_A': mean_squared_error(y_test, grid_search_A.best_estimator_.predict(X_test)),
    'Pipeline_B': mean_squared_error(y_test, grid_search_B.best_estimator_.predict(X_test)),
    'Pipeline_C': mean_squared_error(y_test, grid_search_C.best_estimator_.predict(X_test)),
    'Pipeline_D': mean_squared_error(y_test, grid_search_D.best_estimator_.predict(X_test))
}

# Determining the best model with lowest MSE
best_pipeline_name = min(test_scores, key=test_scores.get)
best_mse = test_scores[best_pipeline_name]
best_rmse = np.sqrt(best_mse)

print(f"Best performing model: {best_pipeline_name}")
print(f"Best RMSE: {best_rmse:.4f}")

Best performing model: Pipeline_B
Best RMSE: 1.4685


# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.